In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import pylab as pl
import numpy as np
%matplotlib inline

In [3]:
from utils import load_data
from preprocessing import (
    drop_unnamed_column,
    filter_open_stores,
    drop_open_column,
    drop_date_column,
    encode_state_holiday,
    drop_duplicates,
)

from functions import (
    split_data,
    scale_feature,
    evaluate_baseline,
    train_and_evaluate_model,
    tune_random_forest,
    compare_train_validation,
    predict_sales,
)

Step 1:  Problem Definition      ✅ done!
Step 2:  Data Collection         ✅ done!
Step 3:  EDA                     → explore, understand, identify issues
Step 4:  Data Splitting          → Train/Test (stratify not needed, Regression)
Step 5:  Data Cleaning           → missing values, wrong types, outliers
Step 6:  Baseline Model          → always predict mean → reference point
Step 7:  Feature Engineering     → encoding, scaling, new features
Step 8:  Feature Selection       → drop useless/correlated features
Step 10: Train Models            → Linear, Random Forest, XGBoost...
Step 11: Hyperparameter Tuning   → tune best models
Step 12: Final Evaluation        → R² on test set → groupX.txt
         +
         Wait for Inference Data → predict → groupX.csv

Target:  Predict Sales  →  a number  →  Regression! ✅
Metric:  R²  ← explicitly required in the challenge
Extra:   MAE/RMSE for better understanding

Output:  groupX.csv  ← all columns + sales prediction column
         groupX.txt  ← R² score from test set

In [4]:
from utils import load_data

df = load_data("sales.csv")
df.head()

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,425390,366,4,2013-04-18,517,1,0,0,0,4422
1,291687,394,6,2015-04-11,694,1,0,0,0,8297
2,411278,807,4,2013-08-29,970,1,1,0,0,9729
3,664714,802,2,2013-05-28,473,1,1,0,0,6513
4,540835,726,4,2013-10-10,1068,1,1,0,0,10882


In [5]:
df.describe()

,Unnamed: 0,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,sales
count,640840.000000,640840.000000,640840.000000,640840.000000,640840.000000,640840.000000,640840.000000,640840.000000
mean,355990.675084,558.211348,4.000189,633.398577,0.830185,0.381718,0.178472,5777.469011
std,205536.290268,321.878521,1.996478,464.094416,0.375470,0.485808,0.382910,3851.338083
min,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,178075.750000,280.000000,2.000000,405.000000,1.000000,0.000000,0.000000,3731.000000
50%,355948.500000,558.000000,4.000000,609.000000,1.000000,0.000000,0.000000,5746.000000
75%,533959.250000,837.000000,6.000000,838.000000,1.000000,1.000000,0.000000,7860.000000
max,712044.000000,1115.000000,7.000000,5458.000000,1.000000,1.000000,1.000000,41551.000000


In [6]:
df.shape

(640840, 10)

In [7]:
df.dtypes

Unnamed: 0              int64
store_ID                int64
day_of_week             int64
date                   object
nb_customers_on_day     int64
open                    int64
promotion               int64
state_holiday          object
school_holiday          int64
sales                   int64
dtype: object

In [8]:
# Missing values
print("Missing values per column:")
print(df.isna().sum())
print()
print("Total missing:", df.isna().sum().sum())
print()

Missing values per column:
Unnamed: 0             0
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
sales                  0
dtype: int64

Total missing: 0



In [9]:
# All rows where sales = 0
closed = df[df['sales'] == 0]

print("Total rows with sales=0:", len(closed))
print()
print("Are they all closed (open=0)?")
print(closed['open'].value_counts())
print()
print("Day of week distribution:")
print(closed['day_of_week'].value_counts().sort_index())
print()
print("Sample rows:")
closed.head(10)

Total rows with sales=0: 108855

Are they all closed (open=0)?
open
0    108824
1        31
Name: count, dtype: int64

Day of week distribution:
day_of_week
1     4517
2     1091
3     2379
4     7093
5     4564
6      449
7    88762
Name: count, dtype: int64

Sample rows:


,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
6,600327,659,7,2014-06-08,0,0,0,0,0,0
10,561067,273,7,2014-10-05,0,0,0,0,0,0
18,409022,767,7,2013-01-27,0,0,0,0,0,0
34,605423,534,7,2014-06-08,0,0,0,0,0,0
35,231682,514,7,2014-03-09,0,0,0,0,0,0
39,357935,1043,7,2013-08-18,0,0,0,0,0,0
44,366083,1093,7,2013-10-06,0,0,0,0,0,0
50,586075,864,7,2013-09-01,0,0,0,0,0,0
56,350680,929,7,2015-02-01,0,0,0,0,0,0
58,561112,532,7,2013-02-24,0,0,0,0,0,0


In [10]:
# Rows where open=1
open_stores = df[df['open'] == 1]

print("Total open rows:", len(open_stores))
print()
print("Sales stats for open stores:")
print(open_stores['sales'].describe())
print()
print("Any sales=0 when open=1?")
print((open_stores['sales'] == 0).sum())
print()
print("Sample:")
open_stores[['open', 'sales']].head(30)

Total open rows: 532016

Sales stats for open stores:
count    532016.000000
mean       6959.251679
std        3105.241710
min           0.000000
25%        4861.000000
50%        6372.000000
75%        8365.000000
max       41551.000000
Name: sales, dtype: float64

Any sales=0 when open=1?
31

Sample:


,open,sales
0,1,4422
1,1,8297
2,1,9729
3,1,6513
4,1,10882
5,1,8406
7,1,11162
8,1,5559
9,1,3997
11,1,6267


In [11]:
df = load_data("sales.csv")

print("Open=1 rows:", (df["open"] == 1).sum())
print("Open=0 rows:", (df["open"] == 0).sum())
print("Total rows:", len(df))

df = filter_open_stores(df)

Open=1 rows: 532016
Open=0 rows: 108824
Total rows: 640840


In [12]:
#unique score : 
print("Unique sales values:", df['sales'].nunique())
print()
print("Sales value counts (top 20):")
print(df['sales'].value_counts().head(20))
print()
print("How many times sales=0:")
print((df['sales'] == 0).sum())

Unique sales values: 20129

Sales value counts (top 20):
sales
5674    146
6049    134
5449    130
5723    128
5584    127
6214    127
5316    126
5044    126
5680    126
5931    126
6171    126
4842    125
5093    123
5460    123
6439    122
5342    122
5553    121
5194    121
5989    121
5056    121
Name: count, dtype: int64

How many times sales=0:
31


In [13]:
print("Open=1 rows:", (df['open'] == 1).sum())
print("Open=0 rows:", (df['open'] == 0).sum())
print("Total rows:", len(df))

#Open=0: 108,824 rows  ← closed stores → drop!
#Open=1: 532,016 rows  ← open stores → keep!
#Total:  640,840 rows

#sales=0: 108,855 rows
#open=0:  108,824 rows
#108,824 → closed (open=0) AND sales=0  ✅ expected
#31      → open=1 BUT sales=0           ⚠️ anomaly!




Open=1 rows: 532016
Open=0 rows: 0
Total rows: 532016


In [14]:
# Open but no sales
anomalies = df[(df['open'] == 1) & (df['sales'] == 0)]
print("Open=1 but sales=0:", len(anomalies))
print()
print(anomalies)

# Drop them → likely data errors ✅

Open=1 but sales=0: 31

        Unnamed: 0  store_ID  day_of_week        date  nb_customers_on_day  \
54379       392697       339            3  2013-01-30                    0   
61554       194070      1039            3  2013-07-10                    0   
63404       573822       983            5  2014-01-17                    0   
82644       540904       387            4  2014-07-24                    0   
117219      432135       762            4  2013-01-17                    0   
137903      236556       303            4  2014-07-24                    0   
163742      253274        25            3  2014-02-12                    0   
180912      502164       835            4  2014-09-11                    0   
188539      389729       835            3  2014-09-10                    0   
209801      581607       548            5  2014-09-05                    0   
235104       91884       623            6  2014-01-25                    0   
238863      143926      1017            

# Step 3:  EDA → explore, understand, identify issues

    # Understanding the fEATURES :
    store_ID             → which store (1-1115)
    day_of_week          → 1-7
    date                 → date as string 
    nb_customers_on_day  → number of customers
    open                 → 0=closed, 1=open
    promotion            → 0/1
    state_holiday        → '0','a','b','c'
    school_holiday       → 0/1
    sales                → TARGET 🎯


    # Identified Issues 

    
    #Encoding / Convert
    date : string - datetime conversation
    state_holiday : needs encoding 

    #Drop issues (Before Spliting)
    sales = 0     → 108,855 rows where store is CLOSED (open=0)
        → remove these
    unamed Column should be removed
    No missing values ✅

    #Scales issues (after Spliting)
    
    Store Id / day_of_week.. scale ? 

#Issues found:
1. Unnamed column    → useless index column
2. open=0 rows       → 108,824 closed store rows
3. sales=0, open=1   → 31 anomaly rows
4. date              → wrong type (string) + needs extraction
5. state_holiday     → categorical ('0','a','b','c') → needs encoding
6. open column       → constant after filtering → useless

#Action Plan before Splitting:

Step 1: Drop Unnamed column ✅
Step 2: Filter → keep only open=1 rows ✅
#Step 3: Drop the 31 anomaly rows (sales=0, open=1)
Step 4: Drop open column (constant now) ✅
Step 5: drop date ✅
Step 6: Encode state_holiday ('0','a','b','c' → numbers) --- Binary / Not One-Hot ✅
Step 7: Drop duplicates ✅


Step X — Spliting ✅


#After Cleaning → Split → Then: ✅

Step 7: Scaling  → fit ONLY on train! ✅
Step 8: Baseline → predict mean of train sales
Step 9: Train models → Linear, Random Forest, XGBoost
Step 10: Evaluate → R² on test set

In [15]:
#Step 1: Drop Unnamed column
from preprocessing import drop_unnamed_column

df = drop_unnamed_column(df)

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

Columns: ['store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'open', 'promotion', 'state_holiday', 'school_holiday', 'sales']
Shape: (532016, 9)


In [16]:
# Script if the passed shop is clsed -> 0 , if oPEN -> USE OUR Model
# Step 2: Filter → keep only open=1 rows
# 
from preprocessing import filter_open_stores

df = filter_open_stores(df)

print("Shape after filtering:", df.shape)
print("Open values:", df["open"].value_counts())

Shape after filtering: (532016, 9)
Open values: open
1    532016
Name: count, dtype: int64


In [17]:
df.head()

,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,366,4,2013-04-18,517,1,0,0,0,4422
1,394,6,2015-04-11,694,1,0,0,0,8297
2,807,4,2013-08-29,970,1,1,0,0,9729
3,802,2,2013-05-28,473,1,1,0,0,6513
4,726,4,2013-10-10,1068,1,1,0,0,10882


In [18]:
from preprocessing import drop_open_column

df = drop_open_column(df)

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

Columns: ['store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'promotion', 'state_holiday', 'school_holiday', 'sales']
Shape: (532016, 8)


In [19]:
from preprocessing import drop_date_column

df = drop_date_column(df)

# Verify
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

Columns: ['store_ID', 'day_of_week', 'nb_customers_on_day', 'promotion', 'state_holiday', 'school_holiday', 'sales']
Shape: (532016, 7)


In [20]:
# Check unique values
print("Unique values in state_holiday:")
print(df['state_holiday'].unique())
print()
print("Value counts:")
print(df['state_holiday'].value_counts())

Unique values in state_holiday:
['0' 'a' 'b' 'c']

Value counts:
state_holiday
0    531437
a       429
b       102
c        48
Name: count, dtype: int64


In [21]:
from preprocessing import encode_state_holiday

df = encode_state_holiday(df)

# Verify
print("Unique values:", df["state_holiday"].unique())
print("Value counts:")
print(df["state_holiday"].value_counts())

Unique values: [0 1]
Value counts:
state_holiday
0    531437
1       579
Name: count, dtype: int64


In [22]:
df.dtypes

store_ID               int64
day_of_week            int64
nb_customers_on_day    int64
promotion              int64
state_holiday          int64
school_holiday         int64
sales                  int64
dtype: object

In [23]:
from functions import split_data

X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

X_train: (372411, 6)
X_val:   (79802, 6)
X_test:  (79803, 6)


In [24]:
from functions import evaluate_baseline

baseline_prediction, baseline_predictions, baseline_mae, baseline_r2 = (
    evaluate_baseline(
        y_train,
        y_val
    )
)

print(f"Baseline prediction (mean): {baseline_prediction:.2f}")
print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline R²: {baseline_r2:.4f}")

Baseline prediction (mean): 6957.20
Baseline MAE: 2294.77
Baseline R²: -0.0000


In [25]:
# Check duplicates
print("Total rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print()

# Check duplicates except target
print("Duplicate features (ignoring sales):")
print(df.duplicated(subset=df.columns.drop('sales')).sum())

Total rows: 532016
Duplicate rows: 68

Duplicate features (ignoring sales):
43061


In [26]:
from preprocessing import drop_duplicates

# Drop duplicates
df = drop_duplicates(df)

# Verify
print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (531948, 7)


In [27]:
from functions import split_data

# Re-split with clean data
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

X_train: (372363, 6)
X_val:   (79792, 6)
X_test:  (79793, 6)


In [28]:
from functions import scale_feature

X_train, X_val, X_test, scaler = scale_feature(
    X_train,
    X_val,
    X_test,
    "store_ID"
)

# Verify
print(f"Mean learned: {scaler.mean_[0]:.2f}")
print(f"Std learned: {scaler.scale_[0]:.2f}")

print("\nBefore scaling: min=1, max=1115")
print("After scaling:")
print(X_train[["store_ID"]].describe())

Mean learned: 557.91
Std learned: 321.67

Before scaling: min=1, max=1115
After scaling:
           store_ID
count  3.723630e+05
mean  -1.226590e-16
std    1.000001e+00
min   -1.731329e+00
25%   -8.639753e-01
50%   -2.839294e-03
75%    8.645143e-01
max    1.731868e+00


In [29]:
# Make sure no sales-related columns leaked in X
print("X_train columns:")
print(X_train.columns.tolist())

# sales should NOT be in X!
assert 'sales' not in X_train.columns
print("✅ No data leakage detected!")

X_train columns:
['store_ID', 'day_of_week', 'nb_customers_on_day', 'promotion', 'state_holiday', 'school_holiday']
✅ No data leakage detected!


### Data Leakage Checklist:
* Split before any Fitting 
* Fit() only on X_train
* transform() on all sets with same scaler 
* No Fututre Information in Features 
* Duplicates are removed before split


In [30]:
from functions import evaluate_baseline

baseline_value, baseline_predictions, mae, r2 = evaluate_baseline(
    y_train,
    y_val
)

print(f"Baseline always predicts: {baseline_value:.2f}")
print(f"First 5 predictions: {baseline_predictions[:5]}")

print(f"\nBaseline MAE: {mae:.2f}")
print(f"Baseline R²: {r2:.4f}")

print("\nOur model must beat:")
print("→ R² > 0.0")
print(f"→ MAE < {mae:.2f}")

Baseline always predicts: 6956.24
First 5 predictions: [6956.24411931 6956.24411931 6956.24411931 6956.24411931 6956.24411931]

Baseline MAE: 2299.80
Baseline R²: -0.0000

Our model must beat:
→ R² > 0.0
→ MAE < 2299.80


In [31]:
from sklearn.linear_model import LinearRegression
from functions import train_and_evaluate_model

lr = LinearRegression()

lr, lr_pred, lr_r2, lr_mae = train_and_evaluate_model(
    lr,
    X_train,
    y_train,
    X_val,
    y_val
)

print("=" * 40)
print("1. Linear Regression")
print(f"   R²: {lr_r2:.4f}")
print(f"   MAE: {lr_mae:.2f}")

1. Linear Regression
   R²: 0.7298
   MAE: 1154.94


In [32]:
from sklearn.ensemble import RandomForestRegressor
from functions import train_and_evaluate_model

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf, rf_pred, rf_r2, rf_mae = train_and_evaluate_model(
    rf,
    X_train,
    y_train,
    X_val,
    y_val
)

print("=" * 40)
print("2. Random Forest")
print(f"   R²: {rf_r2:.4f}")
print(f"   MAE: {rf_mae:.2f}")

2. Random Forest
   R²: 0.9362
   MAE: 502.07


In [58]:
from xgboost import XGBRegressor
from functions import train_and_evaluate_model

xgb = XGBRegressor(
    n_estimators=100,
    random_state=42
)

xgb, xgb_pred, xgb_r2, xgb_mae = train_and_evaluate_model(
    xgb,
    X_train,
    y_train,
    X_val,
    y_val
)

print("=" * 40)
print("3. XGBoost")
print(f"   R²: {xgb_r2:.4f}")
print(f"   MAE: {xgb_mae:.2f}")

3. XGBoost
   R²: 0.8685
   MAE: 816.02


In [34]:
print("--- Random Forest Before Tuning ---")
print(f"R²: {rf_r2:.4f}")
print(f"MAE: {rf_mae:.2f}")

--- Random Forest Before Tuning ---
R²: 0.9362
MAE: 502.07


In [35]:
from functions import tune_random_forest

rf_tuned, rf_tuned_pred, best_params, rf_tuned_r2, rf_tuned_mae = (
    tune_random_forest(
        X_train,
        y_train,
        X_val,
        y_val
    )
)

print("Best params:", best_params)

print("\n--- Comparison on Validation ---")
print(f"Before tuning: R²={rf_r2:.4f}  MAE={rf_mae:.2f}")
print(f"After tuning:  R²={rf_tuned_r2:.4f}  MAE={rf_tuned_mae:.2f}")

if rf_tuned_r2 > rf_r2:
    print("\n✅ Tuned model is better → use tuned!")
else:
    print("\n❌ No improvement → keep original model!")

Fitting 2 folds for each of 5 candidates, totalling 10 fits
[CV] END max_depth=10, min_samples_split=5, n_estimators=200; total time= 4.5min
[CV] END max_depth=10, min_samples_split=5, n_estimators=200; total time= 4.5min
[CV] END max_depth=10, min_samples_split=5, n_estimators=300; total time= 6.7min
[CV] END max_depth=10, min_samples_split=5, n_estimators=300; total time= 6.7min
[CV] END max_depth=10, min_samples_split=10, n_estimators=200; total time= 3.8min
[CV] END max_depth=10, min_samples_split=10, n_estimators=200; total time= 3.8min
[CV] END max_depth=20, min_samples_split=10, n_estimators=300; total time=10.1min
[CV] END max_depth=20, min_samples_split=10, n_estimators=300; total time=10.1min
[CV] END max_depth=20, min_samples_split=5, n_estimators=300; total time=10.3min
[CV] END max_depth=20, min_samples_split=5, n_estimators=300; total time=10.3min
Best params: {'n_estimators': 300, 'min_samples_split': 5, 'max_depth': 20}

--- Comparison on Validation ---
Before tuning: R

In [36]:
from functions import compare_train_validation

train_r2, train_mae, val_r2, val_mae = compare_train_validation(
    rf,
    X_train,
    y_train,
    X_val,
    y_val
)

print("--- Random Forest: Train vs Validation ---")
print(f"{'':20} {'R²':>10} {'MAE':>10}")
print(f"{'Train':20} {train_r2:>10.4f} {train_mae:>10.2f}")
print(f"{'Validation':20} {val_r2:>10.4f} {val_mae:>10.2f}")

--- Random Forest: Train vs Validation ---
                             R²        MAE
Train                    0.9904     195.72
Validation               0.9362     502.07


In [37]:
from functions import evaluate_model

results = []

# Baseline
results.append({
    "Model": "Baseline (mean)",
    "R² Train": "-",
    "R² Val": f"{baseline_r2:.4f}",
    "MAE Train": "-",
    "MAE Val": f"{baseline_mae:.2f}"
})

# Linear Regression
lr_metrics = evaluate_model(
    lr,
    X_train,
    y_train,
    X_val,
    y_val
)

results.append({
    "Model": "Linear Regression",
    "R² Train": f"{lr_metrics['R² Train']:.4f}",
    "R² Val": f"{lr_metrics['R² Val']:.4f}",
    "MAE Train": f"{lr_metrics['MAE Train']:.2f}",
    "MAE Val": f"{lr_metrics['MAE Val']:.2f}",
})

# Random Forest
rf_metrics = evaluate_model(
    rf,
    X_train,
    y_train,
    X_val,
    y_val
)

results.append({
    "Model": "Random Forest (default)",
    "R² Train": f"{rf_metrics['R² Train']:.4f}",
    "R² Val": f"{rf_metrics['R² Val']:.4f}",
    "MAE Train": f"{rf_metrics['MAE Train']:.2f}",
    "MAE Val": f"{rf_metrics['MAE Val']:.2f}",
})

# Tuned Random Forest
rf_tuned_metrics = evaluate_model(
    rf_tuned,
    X_train,
    y_train,
    X_val,
    y_val
)

results.append({
    "Model": "Random Forest (tuned)",
    "R² Train": f"{rf_tuned_metrics['R² Train']:.4f}",
    "R² Val": f"{rf_tuned_metrics['R² Val']:.4f}",
    "MAE Train": f"{rf_tuned_metrics['MAE Train']:.2f}",
    "MAE Val": f"{rf_tuned_metrics['MAE Val']:.2f}",
})

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

                  Model R² Train  R² Val MAE Train MAE Val
        Baseline (mean)        - -0.0000         - 2294.77
      Linear Regression   0.7299  0.7298   1153.67 1154.94
Random Forest (default)   0.9904  0.9362    195.72  502.07
  Random Forest (tuned)   0.9487  0.9158    497.60  621.29


In [38]:
from sklearn.ensemble import RandomForestRegressor
from functions import train_and_evaluate_model

rf_final = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_final, rf_test_pred, rf_test_r2, rf_test_mae = train_and_evaluate_model(
    rf_final,
    X_train,
    y_train,
    X_test,
    y_test
)

print("--- 📍 Step 12: Final Evaluation on Test Set ---")
print(f"R²: {rf_test_r2:.4f}")
print(f"MAE: {rf_test_mae:.2f}")

--- 📍 Step 12: Final Evaluation on Test Set ---
R²: 0.9345
MAE: 503.12


In [39]:
from utils import load_data

# Load inference data
df_inference = load_data("REAL_DATA.csv")

# Check it
print("Shape:", df_inference.shape)
print()
print("Columns:", df_inference.columns.tolist())
print()
print(df_inference.head())

Shape: (71205, 9)

Columns: ['index', 'store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'open', 'promotion', 'state_holiday', 'school_holiday']

    index  store_ID  day_of_week        date  nb_customers_on_day  open  \
0  272371       415            7  01/03/2015                    0     0   
1  558468        27            7  29/12/2013                    0     0   
2   76950       404            3  19/03/2014                  657     1   
3   77556       683            2  29/01/2013                  862     1   
4  456344       920            3  19/03/2014                  591     1   

   promotion state_holiday  school_holiday  
0          0             0               0  
1          0             0               0  
2          1             0               0  
3          0             0               0  
4          1             0               0  


In [40]:
from utils import load_data
from functions import predict_sales

# Load inference data
df_inference = load_data(
    "REAL_DATA.csv",
    index_col=0
)

# Predict sales
df_inference = predict_sales(
    rf,
    scaler,
    df_inference
)

# Verify
print(f"Total rows: {len(df_inference)}")
print(f"Sales = 0: {(df_inference['sales'] == 0).sum()}")
print(f"Sales > 0: {(df_inference['sales'] > 0).sum()}")

print()
print(df_inference[["open", "sales"]].head(10))

Total rows: 71205
Sales = 0: 12102
Sales > 0: 59103

        open  sales
index              
272371     0      0
558468     0      0
76950      1   5922
77556      1   7251
456344     1   5926
436466     1   3897
646251     1   4209
650464     1   7443
162710     0      0
100327     1   7964


/Users/felipemartignon/IRONHACK/IA_Eng/6th_week_deployment/deployment-challenge/functions.py:314: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[5922.11       7251.12       5926.97       ... 5871.72333333 6408.09
 8064.128     ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[open_stores.index, "sales"] = open_stores["sales"]


In [41]:
df_inference.head()

,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
index,,,,,,,,,
272371,415,7,01/03/2015,0,0,0,0,0,0
558468,27,7,29/12/2013,0,0,0,0,0,0
76950,404,3,19/03/2014,657,1,1,0,0,5922
77556,683,2,29/01/2013,862,1,0,0,0,7251
456344,920,3,19/03/2014,591,1,1,0,0,5926


In [42]:
# Quick check
print("Shape:", df_inference.shape)
print("Columns:", df_inference.columns.tolist())
print("Missing values:", df_inference.isna().sum().sum())
print("Sales = 0:", (df_inference['sales'] == 0).sum())
print("Sales > 0:", (df_inference['sales'] > 0).sum())
print()
print(df_inference.head(10))

Shape: (71205, 9)
Columns: ['store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'open', 'promotion', 'state_holiday', 'school_holiday', 'sales']
Missing values: 0
Sales = 0: 12102
Sales > 0: 59103

        store_ID  day_of_week        date  nb_customers_on_day  open  \
index                                                                  
272371       415            7  01/03/2015                    0     0   
558468        27            7  29/12/2013                    0     0   
76950        404            3  19/03/2014                  657     1   
77556        683            2  29/01/2013                  862     1   
456344       920            3  19/03/2014                  591     1   
436466       758            4  26/06/2014                  569     1   
646251       563            1  16/02/2015                  321     1   
650464       930            6  22/11/2014                 1367     1   
162710       756            4  04/06/2015                    0     0   
1003

In [43]:

# Verify
print(df_inference.head(10))
print()
print("state_holiday unique:", df_inference['state_holiday'].unique())
print("sales dtype:", df_inference['sales'].dtype)

        store_ID  day_of_week        date  nb_customers_on_day  open  \
index                                                                  
272371       415            7  01/03/2015                    0     0   
558468        27            7  29/12/2013                    0     0   
76950        404            3  19/03/2014                  657     1   
77556        683            2  29/01/2013                  862     1   
456344       920            3  19/03/2014                  591     1   
436466       758            4  26/06/2014                  569     1   
646251       563            1  16/02/2015                  321     1   
650464       930            6  22/11/2014                 1367     1   
162710       756            4  04/06/2015                    0     0   
100327        49            2  13/01/2015                  546     1   

        promotion state_holiday  school_holiday  sales  
index                                                   
272371          0    

In [44]:
from functions import evaluate_model_on_dataset

rf_test_pred, rf_test_r2, rf_test_mae = evaluate_model_on_dataset(
    rf,
    X_test,
    y_test
)

print("--- Final Evaluation on Test Set ---")
print(f"R²: {rf_test_r2:.4f}")
print(f"MAE: {rf_test_mae:.2f}")

--- Final Evaluation on Test Set ---
R²: 0.9345
MAE: 503.12


In [ ]:
from utils import save_score, save_predictions

# Save R² score
#save_score(rf_test_r2, "group3.txt")
#print("✅ group3.txt created!")

# Save predictions
#save_predictions(df_inference, "group3.csv")
#print("✅ group3.csv created!")

✅ group3.txt created!
✅ group3.csv created!


In [61]:
from utils import save_model

save_model(xgb, "models/xgboost.pkl")
save_model(scaler, "scaler.pkl")